# KarstGeophysics Project

## Geode SEG-2 shot gathers, SEG-Y export, wiggle plots, and Charlie-style frequency contours

This notebook reads Geode SEG-2 (`*.dat`) shot gathers, assigns provisional geometry, writes SEG-Y files, and creates:

1. wiggle plots for each shot gather;
2. Charlie-style frequency-vs-offset contour plots; and
3. receiver-coordinate frequency-contour plots for cave-map interpretation.

The frequency plots use a simple FFT-per-trace workflow that follows Charlie Breithaupt's frequency-vs-offset plotting more closely than the earlier generic spectral plot: each trace is demeaned, normalized by its peak-to-peak range, transformed to the frequency domain, then plotted against offset or receiver coordinate.


---

## Updated filter-comparison version

This version compares four processing branches for the Geode shot gathers:

1. Sarah's moving-average low-frequency removal + 9-sample smoothing.
2. Sarah's filters plus an f-k apparent-velocity notch intended to suppress airwave-like energy.
3. A comparable Butterworth bandpass.
4. Butterworth bandpass plus the same f-k notch.

Outputs are written to separate folders so the products can be compared side-by-side.


In [1]:
from pathlib import Path
from datetime import date
import sys
from typing import Literal, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import obspy
from obspy import Stream
from scipy.signal import butter, sosfiltfilt


def find_repo_root(start: Path, required_subdir: str = "lib") -> Path:
    """Find the repository root by walking upward until ``required_subdir`` is found.

    This lets the notebook live anywhere under the repository while importing
    packages from the top-level ``lib/`` directory.
    """
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / required_subdir).is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find a repository root containing {required_subdir!r} above {start}. "
        "Set repo_root manually."
    )


repo_root = find_repo_root(Path.cwd())
lib_dir = repo_root / "lib"
if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

print(f"repo_root = {repo_root}")
print(f"lib_dir   = {lib_dir}")

from segy_tools.gather import (
    plot_wiggle_gather_from_stream,
    stream_to_gather_arrays,
)
from segy_tools.io import write_segy


from segy_tools.spectral import apply_fk_velocity_filter, plot_fk_spectrum


repo_root = /Users/glennthompson/Developer/karstgeo
lib_dir   = /Users/glennthompson/Developer/karstgeo/lib


## User configuration

Update these values for the local machine and the specific subset of field folders to process.

The notebook handles two folder-name conventions used so far:

- `LBS_*`
- `LBSSP_*`

The processing function below can be run for either or both patterns.


In [2]:
DOWNLOAD_DIR = Path("/Volumes/tachyon/LBSSP_DATA/GEODE_DATA")

# Excel workbook containing Jochen's digitized field-note metadata tables.
# The notebook will look this up relative to the repository first, then fall back
# to the notebook working directory / explicit path if needed.
box_folder_root = Path("/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP")
seismic_analysis_dir = box_folder_root / "04_FieldData" 
METADATA_XLSX = seismic_analysis_dir /  "jochen_field_notes_metadata_tables.xlsx"

# Sheets to search for Geode SEG-2 file numbers. Each sheet should contain a
# file_no column. File numbers are matched against datfile.stem, e.g. 3046.dat.
METADATA_SHEETS = [
    "T1_1m_Refraction",
    "T1_2m_Refraction",
    "T1_Streamer_MASW",
    "T1A_Streamer_MASW",
    "T3_1m_Refraction",
    "T4_1m_Refraction",
]

# Folder name patterns to process. Use ["LBSSP_"] for the current Lafayette Blue Springs folders,
# or include "LBS_" if older folders should also be processed.
FOLDER_PREFIXES = ["LBSSP_"]

# Plotting / processing controls.
feet_to_meters = 0.3048
wiggle_scale = 1.0
normalize_wiggles = True

# Use explicit geometry from the spreadsheet for wiggle plots.
# "receiver_x" plots absolute line coordinate and marks source_x_m.
# "offset" plots receiver_x_m - source_x_m and marks zero.
WIGGLE_X_AXIS = "receiver_x"  # "receiver_x" or "offset"
WIGGLE_SCALE_FRACTION = 0.45  # trace half-width as a fraction of median receiver spacing

# Optional bandpass before wiggle plotting. Keep False for raw plots.
APPLY_BANDPASS_FOR_WIGGLES = False
BANDPASS_FREQ_HZ = (200.0, 400.0)

# Frequency-contour controls.
MAKE_FREQUENCY_CONTOURS = True
FREQ_CONTOUR_MAX_HZ = 100.0       # Charlie-style plots commonly used 0--100 Hz.
FREQ_CONTOUR_LEVELS = 14
FREQ_CONTOUR_PLOT_TYPE_CHARLIE = "contour"      # contour, contourf, or pcolormesh
FREQ_CONTOUR_PLOT_TYPE_MAP = "contourf"         # contourf/pcolormesh is better for map-style plots

# Set to a tuple if you want to mark suspected cave center(s), e.g. (122.0, 130.0).
CAVE_MARKERS_M = (122.0, 130.0)

# File writing controls.
WRITE_SEGY = True
OVERWRITE_PLOTS = True
OVERWRITE_SEGY = False


# -----------------------------------------------------------------------------
# Filter-comparison controls
# -----------------------------------------------------------------------------
# The Geode sample rate should be read from each SEG-2 trace, but this value is
# used as a fallback and for documenting Sarah's intended 160-sample / 20-ms
# low-frequency-removal window.
GEODE_SAMPLE_RATE_FALLBACK_HZ = 8000.0  # set to 8192.0 if headers are wrong
FORCE_SAMPLE_RATE_HZ = None             # None = trust SEG-2 headers; or set 8000.0 / 8192.0

# Sarah's suggested time-domain filters.
SARAH_LOW_REMOVAL_WINDOW_S = 0.020      # 20 ms; 160 samples at 8000 Hz; 164 at 8192 Hz
SARAH_HIGH_SMOOTH_NSAMPLES = 9          # replace each point with average of nearest 9 samples

# Comparable Butterworth bandpass.  The moving-average high-pass corner is roughly
# 0.443 / 0.020 ≈ 22 Hz.  A 9-sample moving average has a rough -3 dB low-pass
# corner of ~394 Hz at 8000 Hz, or ~403 Hz at 8192 Hz.  These are therefore
# reasonable first-pass Butterworth comparison values.
BUTTERWORTH_BANDPASS_HZ = (25.0, 400.0)
BUTTERWORTH_CORNERS = 4

# Optional f-k filtering using segy_tools.spectral.apply_fk_velocity_filter.
# That function is a fan/high-velocity-pass filter: it rejects apparent velocities
# below min_velocity_mps, with an optional raised-cosine transition. It is not a
# narrow 310--370 m/s notch. To target airwave-like energy around 310--370 m/s,
# use min_velocity=370 m/s and a taper width of about 60 m/s. This means energy
# slower than 370 m/s is attenuated, and 370--430 m/s is tapered back in.
# Use cautiously because a loose/unconsolidated near-surface layer may have
# similar apparent velocities to airwaves.
APPLY_FK_BRANCHES = True
FK_MIN_VELOCITY_MPS = 370.0
FK_TAPER_WIDTH_MPS = 60.0
FK_USE_TAPER = True
MAKE_FK_SPECTRUM_PLOTS = True
FK_SPECTRUM_MAX_FREQ_HZ = 800.0

# Branches to generate. Each branch gets its own wiggle/frequency-contour folders.
FILTER_BRANCHES = [
    {"name": "sarah_filters", "filter": "sarah", "fk": False},
    {"name": "sarah_filters_fk_minvel370mps", "filter": "sarah", "fk": True},
    {"name": "butterworth_25_400Hz", "filter": "butter", "fk": False},
    {"name": "butterworth_25_400Hz_fk_minvel370mps", "filter": "butter", "fk": True},
]

# To keep output volume down while testing, optionally set this to an integer.
MAX_FILES_PER_FOLDER = None


### Metadata-driven geometry

This notebook now uses Jochen's digitized field-note workbook as the authoritative geometry lookup.  
Each SEG-2 file is matched by `file_no` (`datfile.stem`) against the configured Excel sheets. Geometry corrections should therefore be made in the spreadsheet rather than hard-coded in this notebook.


## Helper functions

These are project-specific wrappers around `segy_tools`. They mostly encode current field geometry assumptions and folder/file naming conventions. Once the final acquisition tables are settled, these geometry rules should be replaced by metadata-table lookups rather than filename or duration heuristics.


In [3]:
def parse_date_from_geode_folder(folder: Path) -> date | None:
    """Parse a date from a Geode download folder name.

    Expected examples include names like ``LBSSP_MMDDYY_*`` or ``LBS_MMDDYY_*``.
    Returns ``None`` if the folder name does not contain a parseable date token.
    """
    parts = folder.name.split("_")
    if len(parts) < 2:
        return None
    mdy = parts[1]
    if len(mdy) != 6 or not mdy.isdigit():
        return None
    return date(2000 + int(mdy[4:6]), int(mdy[0:2]), int(mdy[2:4]))


def find_geode_folders(download_dir: Path, prefixes=("LBSSP_",)) -> list[Path]:
    """Return Geode data folders whose names start with any requested prefix."""
    folders = []
    for item in sorted(download_dir.iterdir()):
        if item.is_dir() and any(item.name.startswith(prefix) for prefix in prefixes):
            folders.append(item)
    return folders


def resolve_metadata_workbook(path: Path) -> Path:
    """Resolve the metadata workbook path.

    The recommended location is ``repo_root/metadata/jochen_field_notes_metadata_tables.xlsx``.
    If that path does not exist, this function also checks the current working
    directory and the directory containing the notebook output during development.
    """
    candidates = [
        Path(path).expanduser(),
        Path.cwd() / Path(path).name,
        repo_root / Path(path).name,
        repo_root / "metadata" / Path(path).name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find metadata workbook. Checked:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


def _clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with stripped, lower-case, underscore-separated column names."""
    out = df.copy()
    out.columns = [
        str(c).strip().lower().replace(" ", "_").replace("-", "_")
        for c in out.columns
    ]
    return out


def _none_if_nan(value):
    """Convert pandas/NumPy missing values to None."""
    return None if pd.isna(value) else value


def load_geode_metadata_lookup(
    metadata_xlsx: Path,
    sheet_names: list[str] | tuple[str, ...] = METADATA_SHEETS,
) -> pd.DataFrame:
    """Load Jochen field-note metadata tables into one file-number lookup table.

    Parameters
    ----------
    metadata_xlsx
        Excel workbook containing one acquisition table per sheet.

    sheet_names
        Sheet names to scan. Each useful sheet must contain a ``file_no`` column.

    Returns
    -------
    pandas.DataFrame
        Combined metadata table with a ``sheet_name`` column. File numbers are
        stored as integers in the ``file_no`` column.

    Notes
    -----
    The intent is that geometry changes are made in the spreadsheet, not in the
    notebook. This function only standardizes common column names and combines
    sheets into a single lookup table.
    """
    metadata_xlsx = resolve_metadata_workbook(metadata_xlsx)
    rows = []

    xls = pd.ExcelFile(metadata_xlsx)
    available = set(xls.sheet_names)

    for sheet_name in sheet_names:
        if sheet_name not in available:
            print(f"WARNING: metadata sheet not found, skipping: {sheet_name}")
            continue

        df = pd.read_excel(metadata_xlsx, sheet_name=sheet_name)
        df = _clean_column_names(df)

        if "file_no" not in df.columns:
            print(f"WARNING: sheet has no file_no column, skipping: {sheet_name}")
            continue

        df = df[pd.notna(df["file_no"])].copy()
        if df.empty:
            continue

        df["file_no"] = df["file_no"].astype(int)
        df["sheet_name"] = sheet_name

        # Standardize common source-position aliases.
        if "source_position_m" not in df.columns:
            for alias in ("shot_location_m", "shot_position_m", "source_x_m"):
                if alias in df.columns:
                    df["source_position_m"] = df[alias]
                    break

        rows.append(df)

    if not rows:
        raise ValueError(f"No usable metadata rows found in {metadata_xlsx}")

    meta = pd.concat(rows, ignore_index=True, sort=False)

    # Ensure standardized columns exist even where not applicable.
    for col in [
        "source_position_m",
        "receiver_first_m",
        "receiver_last_m",
        "receiver_spacing_m",
        "transect",
        "survey",
        "operator",
        "plate_type",
        "n_blows",
        "n_shots",
        "confidence",
        "review_status",
        "comment",
        "comments",
    ]:
        if col not in meta.columns:
            meta[col] = np.nan

    # Warn about duplicate file numbers, but allow disambiguation by folder/sheet later if desired.
    duplicates = meta.loc[meta["file_no"].duplicated(keep=False), ["file_no", "sheet_name"]]
    if len(duplicates):
        print("WARNING: duplicate file_no values found in metadata lookup:")
        display(duplicates.sort_values("file_no").head(50))

    print(f"Loaded {len(meta)} metadata rows from {metadata_xlsx}")
    return meta


def lookup_geode_metadata(file_no: int | str, metadata_lookup: pd.DataFrame) -> dict:
    """Return one metadata row matching a Geode file number.

    Parameters
    ----------
    file_no
        SEG-2 file number, usually ``int(datfile.stem)``.

    metadata_lookup
        Combined table from :func:`load_geode_metadata_lookup`.

    Returns
    -------
    dict
        Metadata row as a plain dictionary, with missing values converted to None.

    Raises
    ------
    KeyError
        If the file number is absent or ambiguous.
    """
    file_no = int(file_no)
    matches = metadata_lookup.loc[metadata_lookup["file_no"] == file_no].copy()

    if len(matches) == 0:
        raise KeyError(f"No metadata found for file_no={file_no}")

    if len(matches) > 1:
        summary = matches[["file_no", "sheet_name"]].to_dict("records")
        raise KeyError(f"Multiple metadata rows found for file_no={file_no}: {summary}")

    row = matches.iloc[0].to_dict()
    return {key: _none_if_nan(value) for key, value in row.items()}


def metadata_to_geometry(meta: dict, st: Stream) -> dict:
    """Convert one spreadsheet metadata row into the geometry dictionary used downstream.

    Parameters
    ----------
    meta
        Metadata dictionary returned by :func:`lookup_geode_metadata`.

    st
        Raw ObsPy Stream read from the SEG-2 file.

    Returns
    -------
    dict
        Geometry dictionary compatible with plotting/export functions.

    Notes
    -----
    Refraction sheets should ideally provide ``receiver_first_m``,
    ``receiver_last_m`` and ``receiver_spacing_m`` directly. Streamer/MASW sheets
    may lack receiver coordinates; in that case a documented provisional fallback
    assumes 24 channels, 5-ft spacing, and the first active receiver approximately
    30 ft behind the PEG source.
    """
    n_traces = len(st)
    duration_s = max(tr.stats.npts * tr.stats.delta for tr in st)

    sheet_name = meta.get("sheet_name")
    survey = meta.get("survey")
    transect = meta.get("transect")

    source_x_m = meta.get("source_position_m")
    receiver_first_m = meta.get("receiver_first_m")
    receiver_last_m = meta.get("receiver_last_m")
    receiver_spacing_m = meta.get("receiver_spacing_m")

    if source_x_m is None:
        raise ValueError(f"No source_position_m for file_no={meta.get('file_no')} in {sheet_name}")

    # Streamer/MASW fallback when receiver geometry is absent.
    if receiver_first_m is None and sheet_name in {"T1_Streamer_MASW", "T1A_Streamer_MASW"}:
        receiver_spacing_m = 5.0 * feet_to_meters
        # Earlier field notes: first active receiver roughly 30 ft behind PEG.
        receiver_first_m = float(source_x_m) - 30.0 * feet_to_meters
        receiver_last_m = receiver_first_m - (n_traces - 1) * receiver_spacing_m

    # If first receiver is missing but last and spacing exist, infer first.
    if receiver_first_m is None and receiver_last_m is not None and receiver_spacing_m is not None:
        receiver_first_m = float(receiver_last_m) - (n_traces - 1) * float(receiver_spacing_m)

    # If receiver_last is missing but first and spacing exist, infer last.
    if receiver_last_m is None and receiver_first_m is not None and receiver_spacing_m is not None:
        receiver_last_m = float(receiver_first_m) + (n_traces - 1) * float(receiver_spacing_m)

    if receiver_spacing_m is None:
        raise ValueError(f"No receiver_spacing_m for file_no={meta.get('file_no')} in {sheet_name}")

    if receiver_first_m is None:
        raise ValueError(f"No receiver_first_m could be inferred for file_no={meta.get('file_no')} in {sheet_name}")

    network = "GV" if n_traces == 24 else "LB" if n_traces == 72 else "XX"
    geometry_name = survey or sheet_name or "metadata geometry"

    line_min = min(float(receiver_first_m), float(receiver_last_m), float(source_x_m))
    line_max = max(float(receiver_first_m), float(receiver_last_m), float(source_x_m))

    # For receiver-x cave-map plots, show at least the active spread and source.
    line_xlim_m = (line_min, line_max)

    return {
        "file_no": int(meta["file_no"]),
        "sheet_name": sheet_name,
        "transect": transect,
        "survey": survey,
        "geometry_name": geometry_name,
        "network": network,
        "n_traces": n_traces,
        "duration_s": duration_s,
        "receiver_spacing_m": float(receiver_spacing_m),
        "first_receiver_x_m": float(receiver_first_m),
        "last_receiver_x_m": float(receiver_last_m),
        "source_x_m": float(source_x_m),
        "line_xlim_m": line_xlim_m,
        "operator": meta.get("operator"),
        "plate_type": meta.get("plate_type"),
        "n_blows": meta.get("n_blows") or meta.get("n_shots"),
        "confidence": meta.get("confidence"),
        "review_status": meta.get("review_status"),
        "comment": meta.get("comment") or meta.get("comments"),
        "metadata": meta,
    }


def annotate_geode_stream(st: Stream, geom: dict) -> Stream:
    """Assign SEED-style metadata to a Geode SEG-2 stream.

    Geometry values are still supplied as fallback coordinates to `segy_tools`
    plotting and export functions. This function sets useful trace IDs for
    browsing and export.
    """
    st = st.copy()
    for i, tr in enumerate(st):
        tr.stats.station = f"CHA{i + 1:02d}"
        tr.stats.network = geom["network"]
        tr.stats.channel = "DHZ" if tr.stats.sampling_rate < 1000 else "GHZ"
        tr.stats.location = str(geom.get("file_no", 0))[-2:].zfill(2)
    return st


def maybe_bandpass(st: Stream, freqmin: float, freqmax: float) -> Stream:
    """Return a detrended, zero-phase bandpass-filtered copy of a stream."""
    stf = st.copy()
    stf.detrend(type="demean")
    stf.filter("bandpass", freqmin=freqmin, freqmax=freqmax, corners=4, zerophase=True)
    return stf


## Frequency-contour plotting

For each shot gather, the notebook now generates two frequency-domain quick-look plots:

1. **Charlie-style offset plot** — frequency vs signed source-receiver offset. This is closest to Charlie Breithaupt's frequency-vs-offset workflow.
2. **Cave-map receiver-x plot** — frequency vs receiver coordinate along the profile, with suspected cave-center markers shown where relevant.

The FFT workflow follows Charlie's simpler frequency-offset code closely: demean each trace, normalize each trace by its peak-to-peak range, compute a one-sided FFT amplitude spectrum per trace, then normalize the spectral image globally before plotting.


In [4]:
def plot_frequency_contour_from_stream_charlie_fft(
    st: Stream,
    *,
    fallback_receiver_spacing_m: float,
    fallback_first_receiver_x_m: float,
    fallback_source_x_m: float,
    receiver_x_m_override: Sequence[float] | None = None,
    source_x_m_override: float | None = None,
    max_freq_hz: float = 100.0,
    x_axis: Literal["offset", "receiver_x", "absolute_offset", "trace"] = "offset",
    line_xlim_m: tuple[float, float] | None = None,
    trace_scale_m: float | None = None,
    levels: int | Sequence[float] = 14,
    plot_type: Literal["contour", "contourf", "pcolormesh"] = "contour",
    title: str | None = None,
    outfile: Path | None = None,
    close_after_save: bool = False,
    cave_markers_m: tuple[float, ...] = (),
    cmap: str = "viridis",
) -> dict:
    """Create a Charlie-style FFT frequency-vs-position plot from an ObsPy Stream.

    This reproduces the spirit of Charlie Breithaupt's simpler frequency-vs-offset
    workflow:

    1. convert an ObsPy Stream to gather arrays;
    2. force the data to shape ``(n_traces, n_samples)``;
    3. demean each trace;
    4. normalize each trace by its peak-to-peak range, scaled by receiver spacing;
    5. compute an FFT amplitude spectrum for each trace;
    6. normalize the full gather spectrum globally;
    7. plot frequency versus offset or receiver coordinate.

    Parameters
    ----------
    st
        Shot gather as an ObsPy Stream.
    fallback_receiver_spacing_m, fallback_first_receiver_x_m, fallback_source_x_m
        Geometry values used if trace headers do not contain usable coordinates.
    receiver_x_m_override
        Optional explicit receiver coordinates for this gather. Use this when
        SEG-Y/SEG-2 headers are missing, wrong, or only contain relative positions.
    source_x_m_override
        Optional explicit source coordinate for this gather.
    max_freq_hz
        Maximum frequency to show on the vertical axis.
    x_axis
        Plot coordinate system: ``"offset"`` for signed source-receiver offset,
        ``"receiver_x"`` for line coordinate, ``"absolute_offset"`` for absolute
        source-receiver offset, or ``"trace"`` for trace number.
    line_xlim_m
        Optional x-axis limits. Useful for receiver-coordinate plots where the
        full spread or line extent should be visible.
    trace_scale_m
        Scale factor for Charlie-style trace normalization. Defaults to receiver
        spacing, matching the original MATLAB logic ``data = dx * data / range``.
    levels
        Number of contour levels or explicit contour levels.
    plot_type
        ``"contour"`` for line contours, ``"contourf"`` for filled contours,
        or ``"pcolormesh"`` for image-style plotting.
    title
        Optional plot title.
    outfile
        Optional output image path.
    close_after_save
        Close the figure after saving.
    cave_markers_m
        Cave/reference marker positions in receiver-x coordinates. These are
        converted to offset if ``x_axis`` is offset-based.
    cmap
        Matplotlib colormap name.

    Returns
    -------
    dict
        Dictionary containing the plotted arrays, spectra, geometry, figure,
        axis, and colorbar handles.
    """
    if len(st) == 0:
        raise ValueError("Input Stream is empty.")

    time, data, receiver_x_m, source_x_m, geom = stream_to_gather_arrays(
        st,
        sort_by="receiver_x",
        fallback_receiver_spacing_m=fallback_receiver_spacing_m,
        fallback_first_receiver_x_m=fallback_first_receiver_x_m,
        fallback_source_x_m=fallback_source_x_m,
    )

    data = np.asarray(data, dtype=float)
    time = np.asarray(time, dtype=float)
    receiver_x_m = np.asarray(receiver_x_m, dtype=float)

    if receiver_x_m_override is not None:
        receiver_x_m = np.asarray(receiver_x_m_override, dtype=float)

    if source_x_m_override is not None:
        source_x_m = float(source_x_m_override)

    if time.size < 2:
        raise ValueError("Time vector must contain at least two samples.")

    dt = float(np.median(np.diff(time)))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Invalid sample interval.")

    # Standardize data to (n_traces, n_samples).
    if data.shape[0] == receiver_x_m.size:
        working = data.copy()
    elif data.shape[1] == receiver_x_m.size:
        working = data.T.copy()
    else:
        raise ValueError(
            f"Cannot match data shape {data.shape} to "
            f"{receiver_x_m.size} receiver positions."
        )

    # Sort traces by receiver coordinate so contouring behaves correctly.
    idx = np.argsort(receiver_x_m)
    receiver_x_m = receiver_x_m[idx]
    working = working[idx, :]

    # Charlie-style demean and per-trace range normalization.
    working = working - np.nanmean(working, axis=1, keepdims=True)

    if trace_scale_m is None:
        trace_scale_m = fallback_receiver_spacing_m

    trange = np.nanmax(working, axis=1, keepdims=True) - np.nanmin(
        working, axis=1, keepdims=True
    )
    trange[~np.isfinite(trange)] = 1.0
    trange[trange == 0] = 1.0
    working = trace_scale_m * working / trange

    freqs = np.fft.rfftfreq(working.shape[1], d=dt)
    spec = np.abs(np.fft.rfft(working, axis=1))  # (n_traces, n_freq)

    keep = freqs <= max_freq_hz
    freqs = freqs[keep]
    spec = spec[:, keep]

    # Plot as (n_freq, n_traces), matching matplotlib contour expectations.
    spectrum = spec.T

    # Charlie-style global normalization.
    spectrum_norm = spectrum - np.nanmin(spectrum)
    denom = np.nanmax(spectrum_norm)
    if np.isfinite(denom) and denom > 0:
        spectrum_norm = spectrum_norm / denom

    if x_axis == "offset":
        if source_x_m is None:
            raise ValueError("source_x_m required for x_axis='offset'.")
        x = receiver_x_m - float(source_x_m)
        xlabel = "Source-receiver offset (m)"
        marker_positions = tuple(float(m) - float(source_x_m) for m in cave_markers_m)
        source_marker_x = 0.0

    elif x_axis == "absolute_offset":
        if source_x_m is None:
            raise ValueError("source_x_m required for x_axis='absolute_offset'.")
        x = np.abs(receiver_x_m - float(source_x_m))
        order = np.argsort(x)
        x = x[order]
        receiver_x_m = receiver_x_m[order]
        working = working[order, :]
        spec = spec[order, :]
        spectrum_norm = spectrum_norm[:, order]
        xlabel = "Absolute source-receiver offset (m)"
        marker_positions = tuple(abs(float(m) - float(source_x_m)) for m in cave_markers_m)
        source_marker_x = 0.0

    elif x_axis == "receiver_x":
        x = receiver_x_m
        xlabel = "Receiver x (m)"
        marker_positions = cave_markers_m
        source_marker_x = source_x_m

    elif x_axis == "trace":
        x = np.arange(receiver_x_m.size, dtype=float)
        xlabel = "Trace number"
        marker_positions = ()
        source_marker_x = None

    else:
        raise ValueError("Invalid x_axis.")

    fig, ax = plt.subplots(figsize=(8.8, 6.5))

    vmin = float(np.nanmin(spectrum_norm))
    vmax = float(np.nanmax(spectrum_norm))
    if vmax <= vmin:
        vmax = vmin + 1.0

    contour_levels = (
        np.linspace(vmin, vmax, levels)
        if isinstance(levels, int)
        else np.asarray(levels, dtype=float)
    )
    norm = Normalize(vmin=vmin, vmax=vmax)

    if plot_type == "contour":
        cs = ax.contour(
            x,
            freqs,
            spectrum_norm,
            levels=contour_levels,
            cmap=cmap,
            norm=norm,
        )
        # A ScalarMappable gives a stable continuous colorbar for line contours.
        mappable = ScalarMappable(norm=norm, cmap=cmap)
        mappable.set_array([])

    elif plot_type == "contourf":
        cs = ax.contourf(
            x,
            freqs,
            spectrum_norm,
            levels=contour_levels,
            cmap=cmap,
            norm=norm,
        )
        mappable = cs

    elif plot_type == "pcolormesh":
        cs = ax.pcolormesh(
            x,
            freqs,
            spectrum_norm,
            shading="auto",
            cmap=cmap,
            norm=norm,
        )
        mappable = cs

    else:
        raise ValueError("plot_type must be 'contour', 'contourf', or 'pcolormesh'.")

    ax.invert_yaxis()
    ax.set_ylim(max_freq_hz, 0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(title or "Frequency vs offset")
    ax.grid(True, alpha=0.2)

    if line_xlim_m is not None:
        ax.set_xlim(*line_xlim_m)
    else:
        xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
        pad = 0.03 * max(xmax - xmin, 1.0)
        ax.set_xlim(xmin - pad, xmax + pad)

    cbar = fig.colorbar(mappable, ax=ax)
    cbar.set_label("Scaled amplitude")

    xmin, xmax = ax.get_xlim()

    if source_marker_x is not None and xmin <= float(source_marker_x) <= xmax:
        ax.axvline(float(source_marker_x), linestyle=":", linewidth=1.2, label="source")

    for xm in marker_positions:
        if xmin <= float(xm) <= xmax:
            ax.axvline(float(xm), linestyle="--", linewidth=1.2, label=f"marker {xm:g} m")

    handles, labels = ax.get_legend_handles_labels()
    if labels:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), loc="best", fontsize=8)

    fig.tight_layout()

    if outfile is not None:
        outfile = Path(outfile)
        outfile.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outfile, dpi=180, bbox_inches="tight")
        if close_after_save:
            plt.close(fig)

    return {
        "time": time,
        "data": data,
        "processed_data": working,
        "receiver_x_m": receiver_x_m,
        "x": x,
        "x_axis": x_axis,
        "source_x_m": source_x_m,
        "frequencies": freqs,
        "raw_spectrum": spec,
        "spectrum": spectrum_norm,
        "geometry": geom,
        "figure": fig,
        "axis": ax,
        "colorbar": cbar,
    }


## Filter comparison helpers

These functions implement Sarah's two moving-average filters, a comparable Butterworth bandpass, and optional f-k filtering.

Sarah's filters are interpreted as:

- subtract a centered 20 ms moving average from each trace to remove low frequencies;
- then apply a centered 9-sample moving average to remove high frequencies.

For 8000 Hz sampling, the 20 ms window is 160 samples. For 8192 Hz it is about 164 samples. The notebook computes this from the actual trace sampling rate unless `FORCE_SAMPLE_RATE_HZ` is set.

The f-k branch now uses the existing project code:

```python
segy_tools.spectral.apply_fk_velocity_filter
```

That function rejects apparent velocities below a threshold. Here we use `FK_MIN_VELOCITY_MPS = 370` with a `60 m/s` taper as a practical way to suppress airwave-like 310--370 m/s energy, while documenting that this is not a narrow notch and may also remove low-velocity ground roll in the loose surface layer.


In [5]:

def _trace_sample_rate_hz(tr) -> float:
    """Return the sample rate to use for filtering one trace."""
    if FORCE_SAMPLE_RATE_HZ is not None:
        return float(FORCE_SAMPLE_RATE_HZ)
    sr = float(getattr(tr.stats, "sampling_rate", 0.0) or 0.0)
    if sr <= 0:
        sr = float(GEODE_SAMPLE_RATE_FALLBACK_HZ)
    return sr


def centered_moving_average(x: np.ndarray, nsamples: int) -> np.ndarray:
    """Centered moving average with edge padding to avoid large edge transients."""
    x = np.asarray(x, dtype=float)
    nsamples = int(max(1, nsamples))
    if nsamples == 1 or x.size == 0:
        return x.copy()

    # Use an odd window for symmetric centering. If Sarah's nominal value is even
    # (160 at 8000 Hz), add one sample so the output is centered on each sample.
    if nsamples % 2 == 0:
        nsamples += 1

    half = nsamples // 2
    kernel = np.ones(nsamples, dtype=float) / nsamples
    padded = np.pad(x, (half, half), mode="edge")
    return np.convolve(padded, kernel, mode="valid")


def apply_sarah_filters_to_trace_data(x: np.ndarray, sample_rate_hz: float) -> np.ndarray:
    """Apply Sarah's low-frequency removal followed by 9-sample smoothing."""
    x = np.asarray(x, dtype=float)
    low_nsamp = int(round(SARAH_LOW_REMOVAL_WINDOW_S * sample_rate_hz))
    trend = centered_moving_average(x, low_nsamp)
    highpassed = x - trend
    smoothed = centered_moving_average(highpassed, SARAH_HIGH_SMOOTH_NSAMPLES)
    return smoothed


def apply_sarah_filters(st: Stream) -> Stream:
    """Return a copy filtered using Sarah's two moving-average operations."""
    out = st.copy()
    for tr in out:
        sr = _trace_sample_rate_hz(tr)
        tr.data = apply_sarah_filters_to_trace_data(tr.data, sr).astype(np.float32)
    return out


def apply_butterworth_bandpass(st: Stream, freqmin: float, freqmax: float) -> Stream:
    """Return a copy filtered with a zero-phase Butterworth bandpass."""
    out = st.copy()
    for tr in out:
        sr = _trace_sample_rate_hz(tr)
        nyq = 0.5 * sr
        hi = min(float(freqmax), 0.95 * nyq)
        lo = max(float(freqmin), 0.001)
        if lo >= hi:
            raise ValueError(f"Invalid Butterworth band for sr={sr}: {lo}--{hi} Hz")
        sos = butter(BUTTERWORTH_CORNERS, [lo / nyq, hi / nyq], btype="bandpass", output="sos")
        x = np.asarray(tr.data, dtype=float)
        x = x - np.nanmean(x)
        tr.data = sosfiltfilt(sos, x).astype(np.float32)
    return out


def stream_to_data_matrix_in_current_order(st: Stream) -> tuple[np.ndarray, float]:
    """Return data matrix shaped (n_traces, n_samples), trimmed to common length."""
    if len(st) == 0:
        raise ValueError("Empty stream")
    nmin = min(int(tr.stats.npts) for tr in st)
    data = np.vstack([np.asarray(tr.data[:nmin], dtype=float) for tr in st])
    sr = _trace_sample_rate_hz(st[0])
    return data, sr



def _receiver_spacing_from_geom(geom: dict, default: float = 1.0) -> float:
    """Infer receiver spacing for f-k filtering."""
    for key in ["receiver_spacing_m", "dx_m", "spacing_m", "geophone_spacing_m"]:
        try:
            val = float(geom.get(key, np.nan))
            if np.isfinite(val) and val > 0:
                return val
        except Exception:
            pass

    # If receiver positions are available, use median spacing.
    for key in ["receiver_x_m", "x_m", "offsets_m"]:
        vals = geom.get(key)
        if vals is None:
            continue
        try:
            arr = np.asarray(vals, dtype=float)
            arr = arr[np.isfinite(arr)]
            if arr.size >= 2:
                diffs = np.diff(np.sort(arr))
                diffs = diffs[diffs > 0]
                if diffs.size:
                    return float(np.nanmedian(diffs))
        except Exception:
            pass

    return float(default)


def apply_project_fk_velocity_filter(st: Stream, geom: dict) -> Stream:
    """Apply the existing segy_tools.spectral f-k velocity filter.

    This calls:

        apply_fk_velocity_filter(
            st,
            min_velocity_mps=FK_MIN_VELOCITY_MPS,
            receiver_spacing_m=dx,
            use_taper=FK_USE_TAPER,
            taper_width_mps=FK_TAPER_WIDTH_MPS,
        )

    Important: this is a slow-velocity fan mute/high-velocity-pass filter. It is
    not a narrow 310--370 m/s notch. With min_velocity=370 and taper_width=60,
    velocities below 370 m/s are attenuated and 370--430 m/s are tapered.
    """
    if len(st) < 2:
        return st.copy()

    dx = _receiver_spacing_from_geom(geom, default=1.0)

    return apply_fk_velocity_filter(
        st,
        min_velocity_mps=float(FK_MIN_VELOCITY_MPS),
        receiver_spacing_m=float(dx),
        use_taper=bool(FK_USE_TAPER),
        taper_width_mps=float(FK_TAPER_WIDTH_MPS),
    )

def apply_filter_branch(st: Stream, geom: dict, branch: dict) -> Stream:
    """Apply one named filter branch to a stream."""
    kind = branch.get("filter")
    if kind == "sarah":
        out = apply_sarah_filters(st)
    elif kind == "butter":
        out = apply_butterworth_bandpass(st, *BUTTERWORTH_BANDPASS_HZ)
    elif kind in {None, "raw"}:
        out = st.copy()
    else:
        raise ValueError(f"Unknown filter branch kind: {kind}")

    if branch.get("fk", False):
        out = apply_project_fk_velocity_filter(out, geom)

    return out


def branch_description(branch: dict) -> str:
    kind = branch.get("filter")
    if kind == "sarah":
        desc = f"Sarah moving-average filters: subtract {SARAH_LOW_REMOVAL_WINDOW_S*1000:g} ms centered mean; then {SARAH_HIGH_SMOOTH_NSAMPLES}-sample centered mean"
    elif kind == "butter":
        desc = f"Butterworth bandpass {BUTTERWORTH_BANDPASS_HZ[0]:g}-{BUTTERWORTH_BANDPASS_HZ[1]:g} Hz, corners={BUTTERWORTH_CORNERS}"
    else:
        desc = "Raw/unfiltered"
    if branch.get("fk", False):
        desc += f"; segy_tools f-k velocity filter: mute <{FK_MIN_VELOCITY_MPS:g} m/s, taper {FK_TAPER_WIDTH_MPS:g} m/s"
    else:
        desc += "; no f-k notch"
    return desc


## Batch processing with filter-comparison branches

This cell reads each SEG-2 `*.dat` file, assigns geometry from the Excel metadata, and generates four sets of products:

```text
<folder>/filter_comparison/sarah_filters/
<folder>/filter_comparison/sarah_filters_fk_minvel370mps/
<folder>/filter_comparison/butterworth_25_400Hz/
<folder>/filter_comparison/butterworth_25_400Hz_fk_minvel370mps/
```

Each branch has its own `wiggle_plots/` and `frequency_contours/` subfolders. SEG-Y export, if enabled, is still written once per original file to `<folder>/segy/`.

The f-k branches use `segy_tools.spectral.apply_fk_velocity_filter`, not the earlier placeholder notebook implementation.


## Explicit wiggle plotting fix

The original `10_*` notebook produced correct shot gathers because the plotting stage received the correct receiver geometry. In the filter-comparison workflow, filtered/f-k streams may not preserve usable coordinate headers, so the wiggle plots are now drawn with receiver positions computed directly from the Excel metadata:

```python
receiver_x = linspace(first_receiver_x_m, last_receiver_x_m, n_traces)
```

This prevents all traces from being centered at `x=0` and keeps the source marker consistent with the chosen x-axis.


In [6]:

def receiver_positions_from_geometry(geom: dict, n_traces: int) -> np.ndarray:
    """Return receiver x positions directly from spreadsheet-derived geometry.

    This avoids relying on SEG-2/SEG-Y trace headers or library fallbacks when
    plotting filtered streams. It preserves reversed spreads by using
    first_receiver_x_m and last_receiver_x_m when both are available.
    """
    first = float(geom["first_receiver_x_m"])
    spacing = abs(float(geom["receiver_spacing_m"]))

    last = geom.get("last_receiver_x_m", None)
    if last is not None and np.isfinite(float(last)):
        last = float(last)
        if n_traces > 1:
            return np.linspace(first, last, n_traces)
        return np.array([first], dtype=float)

    return first + np.arange(n_traces, dtype=float) * spacing


def plot_wiggle_gather_explicit_geometry(
    st: Stream,
    geom: dict,
    *,
    tmin: float = 0.0,
    tmax: float | None = None,
    scale_fraction: float = 0.45,
    normalize: bool = True,
    x_axis: str = "receiver_x",
    title: str | None = None,
    outfile: Path | None = None,
    close_after_save: bool = True,
):
    """Plot a wiggle gather using explicit spreadsheet receiver coordinates.

    This is intentionally independent of ``segy_tools.gather.plot_wiggle_gather_from_stream``
    because the filter-comparison branches can produce streams whose headers do
    not preserve usable receiver coordinates. The original 10_* notebook worked
    because the plotting stage got the correct geometry; this function makes that
    geometry use explicit and unambiguous.
    """
    if len(st) == 0:
        raise ValueError("Cannot plot empty Stream")

    n_traces = len(st)
    receiver_x = receiver_positions_from_geometry(geom, n_traces)
    source_x = float(geom["source_x_m"])

    if x_axis == "receiver_x":
        x0 = receiver_x.copy()
        source_line_x = source_x
        xlabel = "Receiver x (m)"
        cave_positions = list(CAVE_MARKERS_M)
    elif x_axis == "offset":
        x0 = receiver_x - source_x
        source_line_x = 0.0
        xlabel = "Offset from source (m)"
        cave_positions = [c - source_x for c in CAVE_MARKERS_M]
    else:
        raise ValueError("x_axis must be 'receiver_x' or 'offset'")

    # Use common trimmed time/data arrays.
    sr = float(st[0].stats.sampling_rate)
    nmin = min(int(tr.stats.npts) for tr in st)
    if tmax is None:
        tmax = nmin / sr

    i0 = max(0, int(round(tmin * sr)))
    i1 = min(nmin, int(round(tmax * sr)))
    if i1 <= i0:
        raise ValueError(f"Invalid time window: {tmin} to {tmax} s")

    t = np.arange(i0, i1, dtype=float) / sr

    data = []
    for tr in st:
        x = np.asarray(tr.data[:nmin], dtype=float)[i0:i1]
        x = x - np.nanmean(x)
        if normalize:
            denom = np.nanmax(np.abs(x))
            if np.isfinite(denom) and denom > 0:
                x = x / denom
        data.append(x)
    data = np.asarray(data, dtype=float)

    # Scale traces relative to receiver spacing in the plotted coordinate system.
    finite_x = x0[np.isfinite(x0)]
    if finite_x.size >= 2:
        dx = np.nanmedian(np.abs(np.diff(np.sort(finite_x))))
        if not np.isfinite(dx) or dx == 0:
            dx = abs(float(geom["receiver_spacing_m"]))
    else:
        dx = abs(float(geom["receiver_spacing_m"]))

    amp_scale = float(scale_fraction) * dx

    fig, ax = plt.subplots(figsize=(14, 8))

    for i in range(n_traces):
        xi = x0[i]
        yi = t
        wig = xi + amp_scale * data[i]

        ax.plot(wig, yi, color="black", linewidth=0.45)

        # Fill positive/negative lobes relative to trace center.
        ax.fill_betweenx(
            yi, xi, wig,
            where=(wig >= xi),
            color="red",
            alpha=0.35,
            linewidth=0,
        )
        ax.fill_betweenx(
            yi, xi, wig,
            where=(wig < xi),
            color="blue",
            alpha=0.35,
            linewidth=0,
        )

    ax.axvline(source_line_x, color="tab:blue", linestyle="--", linewidth=1.0, label="source")

    for cave_x in cave_positions:
        ax.axvline(cave_x, color="tab:green", linestyle=":", linewidth=0.9, alpha=0.8)

    ax.set_ylim(tmax, tmin)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Time (s)")
    if title:
        ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")

    # Give traces a little horizontal margin.
    xmin = np.nanmin(x0) - 2.0 * dx
    xmax = np.nanmax(x0) + 2.0 * dx
    if x_axis == "receiver_x":
        # Include source if outside active spread.
        xmin = min(xmin, source_line_x - 2.0 * dx)
        xmax = max(xmax, source_line_x + 2.0 * dx)
    ax.set_xlim(xmin, xmax)

    fig.tight_layout()

    if outfile is not None:
        outfile = Path(outfile)
        outfile.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outfile, dpi=150, bbox_inches="tight")
        if close_after_save:
            plt.close(fig)

    return fig, ax


In [7]:
def _frequency_output_paths(
    freq_contour_dir: Path,
    datfile: Path,
    max_freq_hz: float,
) -> tuple[Path, Path]:
    """Return output paths for Charlie-style and receiver-coordinate frequency plots."""
    freq_charlie_file = (
        freq_contour_dir
        / "charlie_offset"
        / f"{datfile.stem}_charlie_frequency_vs_offset_{max_freq_hz:g}Hz.png"
    )
    freq_map_file = (
        freq_contour_dir
        / "receiver_x_map"
        / f"{datfile.stem}_frequency_vs_receiver_x_{max_freq_hz:g}Hz.png"
    )
    return freq_charlie_file, freq_map_file


def make_frequency_contour_products(
    st: Stream,
    geom: dict,
    *,
    title: str,
    freq_contour_dir: Path,
    datfile: Path,
) -> tuple[Path, Path]:
    """Create Charlie-style and receiver-coordinate frequency contour plots."""
    freq_charlie_file, freq_map_file = _frequency_output_paths(
        freq_contour_dir,
        datfile,
        FREQ_CONTOUR_MAX_HZ,
    )

    for parent in [freq_charlie_file.parent, freq_map_file.parent]:
        parent.mkdir(parents=True, exist_ok=True)

    if OVERWRITE_PLOTS or not freq_charlie_file.exists():
        plot_frequency_contour_from_stream_charlie_fft(
            st,
            fallback_receiver_spacing_m=geom["receiver_spacing_m"],
            fallback_first_receiver_x_m=geom["first_receiver_x_m"],
            fallback_source_x_m=geom["source_x_m"],
            source_x_m_override=geom["source_x_m"],
            max_freq_hz=FREQ_CONTOUR_MAX_HZ,
            x_axis="offset",
            plot_type=FREQ_CONTOUR_PLOT_TYPE_CHARLIE,
            levels=FREQ_CONTOUR_LEVELS,
            title=title,
            outfile=freq_charlie_file,
            close_after_save=True,
        )
    else:
        print(f"  Charlie-style frequency contour exists, skipping: {freq_charlie_file.name}")

    if OVERWRITE_PLOTS or not freq_map_file.exists():
        plot_frequency_contour_from_stream_charlie_fft(
            st,
            fallback_receiver_spacing_m=geom["receiver_spacing_m"],
            fallback_first_receiver_x_m=geom["first_receiver_x_m"],
            fallback_source_x_m=geom["source_x_m"],
            source_x_m_override=geom["source_x_m"],
            max_freq_hz=FREQ_CONTOUR_MAX_HZ,
            x_axis="receiver_x",
            line_xlim_m=geom.get("line_xlim_m", None),
            plot_type=FREQ_CONTOUR_PLOT_TYPE_MAP,
            levels=FREQ_CONTOUR_LEVELS,
            title=title + " — receiver-x cave-map plot",
            outfile=freq_map_file,
            cave_markers_m=CAVE_MARKERS_M,
            close_after_save=True,
        )
    else:
        print(f"  receiver-x frequency contour exists, skipping: {freq_map_file.name}")

    return freq_charlie_file, freq_map_file


def _title_for_gather(data_dir: Path, datfile: Path, geom: dict, branch: dict | None = None) -> str:
    extras = []
    if geom.get("operator"):
        extras.append(f"operator={geom['operator']}")
    if geom.get("n_blows"):
        extras.append(f"{geom['n_blows']:.0f} blows")
    if geom.get("plate_type") and str(geom["plate_type"]) not in {"None", "nan", "metal"}:
        extras.append(str(geom["plate_type"]))
    if branch is not None:
        extras.append(branch_description(branch))

    extra_text = ""
    if extras:
        extra_text = " (" + "; ".join(extras) + ")"

    return (
        f"{data_dir.name}/{datfile.stem}: "
        f"source x={geom['source_x_m']:.1f} m, "
        f"{geom['geometry_name']}"
        f"{extra_text}"
    )


def process_one_branch_for_geode_file(
    st: Stream,
    geom: dict,
    datfile: Path,
    data_dir: Path,
    *,
    branch: dict,
    branch_root: Path,
) -> dict:
    """Create wiggle and frequency-contour outputs for one filter branch."""
    branch_name = branch["name"]
    print(f"  branch: {branch_name}")

    st_branch = apply_filter_branch(st, geom, branch)

    wiggle_dir = branch_root / "wiggle_plots"
    freq_contour_dir = branch_root / "frequency_contours"
    fk_spectrum_dir = branch_root / "fk_spectra"
    for d in [wiggle_dir, freq_contour_dir, fk_spectrum_dir]:
        d.mkdir(parents=True, exist_ok=True)

    wiggle_file = wiggle_dir / f"{datfile.stem}_{branch_name}_shot_gather.png"
    fk_spectrum_file = fk_spectrum_dir / f"{datfile.stem}_{branch_name}_fk_spectrum.png"
    freq_charlie_file, freq_map_file = _frequency_output_paths(
        freq_contour_dir,
        datfile,
        FREQ_CONTOUR_MAX_HZ,
    )

    title = _title_for_gather(data_dir, datfile, geom, branch)

    if OVERWRITE_PLOTS or not wiggle_file.exists():
        plot_wiggle_gather_explicit_geometry(
            st_branch,
            geom,
            tmin=0.0,
            tmax=geom["duration_s"],
            scale_fraction=WIGGLE_SCALE_FRACTION,
            normalize=normalize_wiggles,
            x_axis=WIGGLE_X_AXIS,
            title=title,
            outfile=wiggle_file,
        )
    else:
        print(f"    wiggle exists, skipping: {wiggle_file.name}")

    if MAKE_FREQUENCY_CONTOURS:
        freq_charlie_file, freq_map_file = make_frequency_contour_products(
            st_branch,
            geom,
            title=title,
            freq_contour_dir=freq_contour_dir,
            datfile=datfile,
        )

    if MAKE_FK_SPECTRUM_PLOTS:
        if OVERWRITE_PLOTS or not fk_spectrum_file.exists():
            try:
                fig = plot_fk_spectrum(
                    st_branch,
                    receiver_spacing_m=geom["receiver_spacing_m"],
                    max_display_freq_hz=FK_SPECTRUM_MAX_FREQ_HZ,
                    reference_velocity_mps=FK_MIN_VELOCITY_MPS if branch.get("fk", False) else None,
                    title=title + " — f-k spectrum",
                )
                fig.savefig(fk_spectrum_file, dpi=150, bbox_inches="tight")
                plt.close(fig)
            except Exception as exc:
                print(f"    f-k spectrum plot failed: {exc}")
        else:
            print(f"    f-k spectrum exists, skipping: {fk_spectrum_file.name}")

    return {
        "branch": branch_name,
        "branch_description": branch_description(branch),
        "wiggle_file": str(wiggle_file),
        "freq_charlie_file": str(freq_charlie_file) if MAKE_FREQUENCY_CONTOURS else None,
        "freq_map_file": str(freq_map_file) if MAKE_FREQUENCY_CONTOURS else None,
        "fk_spectrum_file": str(fk_spectrum_file) if MAKE_FK_SPECTRUM_PLOTS else None,
    }


def process_geode_seg2_file(
    datfile: Path,
    *,
    data_dir: Path,
    metadata_lookup: pd.DataFrame,
    branch_base_dir: Path,
    segydir: Path,
) -> list[dict]:
    """Process one Geode SEG-2 file through all configured filter branches."""
    print(f"Processing file: {datfile}")
    st_raw = obspy.read(str(datfile), format="SEG2")

    try:
        file_no = int(datfile.stem)
    except ValueError as exc:
        raise ValueError(f"SEG-2 file stem is not an integer file_no: {datfile.name}") from exc

    meta = lookup_geode_metadata(file_no, metadata_lookup)
    geom = metadata_to_geometry(meta, st_raw)
    st = annotate_geode_stream(st_raw, geom)

    print(
        f"  traces={geom['n_traces']}, duration={geom['duration_s']:.3f} s, "
        f"sr={st[0].stats.sampling_rate:g} Hz, geom={geom['geometry_name']}, "
        f"source_x={geom['source_x_m']:.2f} m"
    )

    # Optional raw SEG-Y export once per original file.
    segy_file = segydir / f"{datfile.stem}.segy"
    if WRITE_SEGY:
        if OVERWRITE_SEGY or not segy_file.exists():
            st.write(str(segy_file), format="SEGY", byteorder=">")
        else:
            print(f"  SEG-Y exists, skipping: {segy_file.name}")

    records = []
    for branch in FILTER_BRANCHES:
        branch_root = branch_base_dir / branch["name"]
        branch_rec = process_one_branch_for_geode_file(
            st,
            geom,
            datfile,
            data_dir,
            branch=branch,
            branch_root=branch_root,
        )

        base_rec = {
            "folder": data_dir.name,
            "datfile": datfile.name,
            "n_traces": geom["n_traces"],
            "duration_s": geom["duration_s"],
            "sample_rate_hz": float(st[0].stats.sampling_rate) if len(st) else np.nan,
            "file_no": geom["file_no"],
            "sheet_name": geom.get("sheet_name"),
            "transect": geom.get("transect"),
            "survey": geom.get("survey"),
            "geometry_name": geom["geometry_name"],
            "receiver_spacing_m": geom["receiver_spacing_m"],
            "first_receiver_x_m": geom["first_receiver_x_m"],
            "last_receiver_x_m": geom.get("last_receiver_x_m"),
            "source_x_m": geom["source_x_m"],
            "n_blows": geom.get("n_blows"),
            "operator": geom.get("operator"),
            "plate_type": geom.get("plate_type"),
            "confidence": geom.get("confidence"),
            "review_status": geom.get("review_status"),
            "comment": geom.get("comment"),
            "segy_file": str(segy_file) if WRITE_SEGY else None,
        }
        base_rec.update(branch_rec)
        records.append(base_rec)

    return records


def process_geode_folders(
    download_dir: Path,
    prefixes=("LBSSP_",),
    *,
    metadata_xlsx: Path = METADATA_XLSX,
    metadata_sheets: list[str] | tuple[str, ...] = METADATA_SHEETS,
) -> pd.DataFrame:
    """Batch-process all matching Geode folders and return a summary table."""
    geode_folders = find_geode_folders(download_dir, prefixes=prefixes)
    print(f"Found {len(geode_folders)} matching Geode folders")

    metadata_lookup = load_geode_metadata_lookup(metadata_xlsx, metadata_sheets)

    records = []
    for day_index, data_dir in enumerate(geode_folders):
        field_date = parse_date_from_geode_folder(data_dir)
        print("\n" + "=" * 80)
        print(f"Folder: {data_dir}")
        print(f"Parsed date: {field_date}")

        alldatfiles = sorted(data_dir.glob("*.dat"))
        if MAX_FILES_PER_FOLDER is not None:
            alldatfiles = alldatfiles[:MAX_FILES_PER_FOLDER]
        print(f"Found {len(alldatfiles)} SEG-2 .dat files to process")

        branch_base_dir = data_dir / "filter_comparison"
        segydir = data_dir / "segy"
        for d in [branch_base_dir, segydir]:
            d.mkdir(exist_ok=True)

        for file_index, datfile in enumerate(alldatfiles):
            try:
                recs = process_geode_seg2_file(
                    datfile,
                    data_dir=data_dir,
                    metadata_lookup=metadata_lookup,
                    branch_base_dir=branch_base_dir,
                    segydir=segydir,
                )
                for rec in recs:
                    rec["field_date"] = field_date.isoformat() if field_date else None
                    records.append(rec)
            except Exception as exc:
                print(f"ERROR processing {datfile}: {exc}")
                records.append({
                    "folder": data_dir.name,
                    "datfile": datfile.name,
                    "error": repr(exc),
                })

    summary = pd.DataFrame(records)
    if len(summary):
        summary_file = download_dir / "geode_filter_comparison_processing_summary.csv"
        summary.to_csv(summary_file, index=False)
        print(f"\nWrote summary: {summary_file}")
    return summary


## Run batch processing

Uncomment and run the cell below when ready. The summary table records the assumed geometry and all output products.


In [ ]:
summary = process_geode_folders(DOWNLOAD_DIR, prefixes=FOLDER_PREFIXES)
summary.head()


Found 2 matching Geode folders
Loaded 327 metadata rows from /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/04_FieldData/jochen_field_notes_metadata_tables.xlsx

Folder: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826
Parsed date: 2026-05-18
Found 86 SEG-2 .dat files to process
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/._3022.dat
ERROR processing /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/._3022.dat: Wrong File Descriptor Block ID
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/._3024.dat
ERROR processing /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/._3024.dat: Wrong File Descriptor Block ID
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3005.dat
  traces=72, duration=0.400 s, sr=8000 Hz, geom=T1_1m_refraction, source_x=82.50 m
  SEG-Y exists, skipping: 3005.segy
  branch: sarah_filters


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:365: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


  branch: sarah_filters_fk_minvel370mps
  branch: butterworth_25_400Hz
  branch: butterworth_25_400Hz_fk_minvel370mps
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3006.dat
  traces=72, duration=0.400 s, sr=8000 Hz, geom=T1_1m_refraction, source_x=84.50 m
  SEG-Y exists, skipping: 3006.segy
  branch: sarah_filters
  branch: sarah_filters_fk_minvel370mps
  branch: butterworth_25_400Hz
  branch: butterworth_25_400Hz_fk_minvel370mps
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3007.dat
  traces=72, duration=0.400 s, sr=8000 Hz, geom=T1_1m_refraction, source_x=86.50 m
  SEG-Y exists, skipping: 3007.segy
  branch: sarah_filters
  branch: sarah_filters_fk_minvel370mps
  branch: butterworth_25_400Hz
  branch: butterworth_25_400Hz_fk_minvel370mps
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3008.dat
  traces=72, duration=0.400 s, sr=8000 Hz, geom=T1_1m_refraction, source_x=88.50 m
  SEG-Y exists, skipping: 3008.segy
  branc

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/segy/core.py:362: UserWarning: CREATING TRACE HEADER
  warnings.warn("CREATING TRACE HEADER")


  branch: sarah_filters_fk_minvel370mps
  branch: butterworth_25_400Hz
  branch: butterworth_25_400Hz_fk_minvel370mps
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051926/4019.dat
  traces=72, duration=0.400 s, sr=8000 Hz, geom=T3_1m_refraction, source_x=35.50 m
  branch: sarah_filters
  branch: sarah_filters_fk_minvel370mps
  branch: butterworth_25_400Hz
  branch: butterworth_25_400Hz_fk_minvel370mps
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051926/4020.dat
  traces=72, duration=0.400 s, sr=8000 Hz, geom=T3_1m_refraction, source_x=37.50 m
  branch: sarah_filters
  branch: sarah_filters_fk_minvel370mps
  branch: butterworth_25_400Hz
  branch: butterworth_25_400Hz_fk_minvel370mps
Processing file: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051926/4021.dat
  traces=72, duration=0.400 s, sr=8000 Hz, geom=T3_1m_refraction, source_x=39.50 m
  branch: sarah_filters
  branch: sarah_filters_fk_minvel370mps
  branch: butterworth_25_400Hz
  branch: butterwort

## Inspect processing summary

Use this cell to identify failed files, check geometry assumptions, and quickly locate generated frequency contour plots.


In [ ]:
if "summary" in globals() and len(summary):
    display(summary)
    if "error" in summary.columns:
        display(summary[summary["error"].notna()])
